In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
df = pd.read_csv(r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\data\raw\mp_data.csv")

print("Original Shape:", df.shape)

df.head()

Original Shape: (2394341, 15)


,State Name,District Name,Market Name,Variety,Group,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date,Commodity_File,Arrivals,Min Price,Max Price,Modal Price
0,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,1.4,2400.0,2400.0,2400.0,31 Jan 2024,Wheat,NaN,NaN,NaN,NaN
1,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,8.0,2510.0,2525.0,2510.0,25 Jan 2024,Wheat,NaN,NaN,NaN,NaN
2,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,1.5,2590.0,2590.0,2590.0,14 Jan 2024,Wheat,NaN,NaN,NaN,NaN
3,Madhya Pradesh,Bhind,Lahar,Other,Cereals,65.7,2315.0,2406.0,2380.0,20 Dec 2023,Wheat,NaN,NaN,NaN,NaN
4,Madhya Pradesh,Bhind,Lahar,Other,Cereals,86.4,2300.0,2391.0,2350.0,19 Dec 2023,Wheat,NaN,NaN,NaN,NaN


In [3]:
df = df[["State Name",
    "District Name",
    "Market Name",
    "Commodity_File",
    "Arrivals (Tonnes)",
    "Min Price (Rs./Quintal)",
    "Max Price (Rs./Quintal)",
    "Modal Price (Rs./Quintal)",
    "Reported Date"]]

df.head()

,State Name,District Name,Market Name,Commodity_File,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date
0,Madhya Pradesh,Bhind,Lahar,Wheat,1.4,2400.0,2400.0,2400.0,31 Jan 2024
1,Madhya Pradesh,Bhind,Lahar,Wheat,8.0,2510.0,2525.0,2510.0,25 Jan 2024
2,Madhya Pradesh,Bhind,Lahar,Wheat,1.5,2590.0,2590.0,2590.0,14 Jan 2024
3,Madhya Pradesh,Bhind,Lahar,Wheat,65.7,2315.0,2406.0,2380.0,20 Dec 2023
4,Madhya Pradesh,Bhind,Lahar,Wheat,86.4,2300.0,2391.0,2350.0,19 Dec 2023


In [4]:
df.columns = ["state",
    "district",
    "market",
    "commodity",
    "arrivals",
    "min_price",
    "max_price",
    "modal_price",
    "date"]

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date
0,Madhya Pradesh,Bhind,Lahar,Wheat,1.4,2400.0,2400.0,2400.0,31 Jan 2024
1,Madhya Pradesh,Bhind,Lahar,Wheat,8.0,2510.0,2525.0,2510.0,25 Jan 2024
2,Madhya Pradesh,Bhind,Lahar,Wheat,1.5,2590.0,2590.0,2590.0,14 Jan 2024
3,Madhya Pradesh,Bhind,Lahar,Wheat,65.7,2315.0,2406.0,2380.0,20 Dec 2023
4,Madhya Pradesh,Bhind,Lahar,Wheat,86.4,2300.0,2391.0,2350.0,19 Dec 2023


In [6]:
df["date"] = pd.to_datetime(
    df["date"],
    format="mixed",
    errors="coerce"
)

# numeric columns
numeric_cols = [
    "arrivals",
    "min_price",
    "max_price",
    "modal_price"
]

# convert numeric columns
for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce")
df = df.dropna()

df = df.reset_index(drop=True)

In [7]:
group_cols = [
    "state",
    "district",
    "market",
    "commodity"
]

df = df.sort_values(
    by=group_cols + ["date"]
)

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20


In [8]:
df["target"] = (

    df.groupby(group_cols)["modal_price"]
    .shift(-1)

)

df[[
    "modal_price",
    "target"
]].head(10)

,modal_price,target
1895193,1600.0,0.0
1895194,0.0,2200.0
1895195,2200.0,1950.0
1895196,1950.0,2033.0
1895197,2033.0,1850.0
1895198,1850.0,2136.0
1895199,2136.0,1850.0
1895200,1850.0,1504.0
1895201,1504.0,1850.0
1895202,1850.0,1978.0


In [9]:
df["day"] = df["date"].dt.day

df["month"] = df["date"].dt.month

df["week"] = (
    df["date"]
    .dt
    .isocalendar()
    .week
    .astype(int)
)

df["quarter"] = df["date"].dt.quarter

df["year"] = df["date"].dt.year

df.head()


,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date,target,day,month,week,quarter,year
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21,0.0,21,3,12,1,2006
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29,2200.0,29,3,13,1,2007
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29,1950.0,29,3,13,1,2007
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20,2033.0,20,4,16,2,2007
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20,1850.0,20,4,16,2,2007


In [10]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df[[
    "month",
    "month_sin",
    "month_cos"
]].head()

,month,month_sin,month_cos
1895193,3,1.000000,6.123234e-17
1895194,3,1.000000,6.123234e-17
1895195,3,1.000000,6.123234e-17
1895196,4,0.866025,-5.000000e-01
1895197,4,0.866025,-5.000000e-01


In [11]:
for lag in [1, 3, 7, 14]:

    df[f"price_lag_{lag}d"] = (

        df.groupby(group_cols)["modal_price"]
        .shift(lag)

    )

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date,target,...,month,week,quarter,year,month_sin,month_cos,price_lag_1d,price_lag_3d,price_lag_7d,price_lag_14d
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21,0.0,...,3,12,1,2006,1.000000,6.123234e-17,NaN,NaN,NaN,NaN
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29,2200.0,...,3,13,1,2007,1.000000,6.123234e-17,1600.0,NaN,NaN,NaN
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29,1950.0,...,3,13,1,2007,1.000000,6.123234e-17,0.0,NaN,NaN,NaN
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20,2033.0,...,4,16,2,2007,0.866025,-5.000000e-01,2200.0,1600.0,NaN,NaN
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20,1850.0,...,4,16,2,2007,0.866025,-5.000000e-01,1950.0,0.0,NaN,NaN


In [12]:
for lag in [1, 7, 14]:

    df[f"arrival_lag_{lag}d"] = (

        df.groupby(group_cols)["arrivals"]
        .shift(lag)

    )

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date,target,...,year,month_sin,month_cos,price_lag_1d,price_lag_3d,price_lag_7d,price_lag_14d,arrival_lag_1d,arrival_lag_7d,arrival_lag_14d
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21,0.0,...,2006,1.000000,6.123234e-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29,2200.0,...,2007,1.000000,6.123234e-17,1600.0,NaN,NaN,NaN,5.0,NaN,NaN
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29,1950.0,...,2007,1.000000,6.123234e-17,0.0,NaN,NaN,NaN,10.6,NaN,NaN
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20,2033.0,...,2007,0.866025,-5.000000e-01,2200.0,1600.0,NaN,NaN,10.6,NaN,NaN
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20,1850.0,...,2007,0.866025,-5.000000e-01,1950.0,0.0,NaN,NaN,16.7,NaN,NaN


In [13]:
for window in [7, 14]:

    df[f"rolling_mean_{window}d"] = (

        df.groupby(group_cols)["modal_price"]

        .transform(
            lambda x: x.rolling(window).mean()
        )

    )

    df[f"rolling_std_{window}d"] = (

        df.groupby(group_cols)["modal_price"]

        .transform(
            lambda x: x.rolling(window).std()
        )

    )

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date,target,...,price_lag_3d,price_lag_7d,price_lag_14d,arrival_lag_1d,arrival_lag_7d,arrival_lag_14d,rolling_mean_7d,rolling_std_7d,rolling_mean_14d,rolling_std_14d
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29,2200.0,...,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29,1950.0,...,NaN,NaN,NaN,10.6,NaN,NaN,NaN,NaN,NaN,NaN
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20,2033.0,...,1600.0,NaN,NaN,10.6,NaN,NaN,NaN,NaN,NaN,NaN
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20,1850.0,...,0.0,NaN,NaN,16.7,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df["price_momentum_7d"] = (
    df["modal_price"] - df["price_lag_7d"]
)

df["arrival_price_ratio"] = (
    df["arrivals"] / (df["modal_price"] + 1)
)

df.head()

,state,district,market,commodity,arrivals,min_price,max_price,modal_price,date,target,...,price_lag_14d,arrival_lag_1d,arrival_lag_7d,arrival_lag_14d,rolling_mean_7d,rolling_std_7d,rolling_mean_14d,rolling_std_14d,price_momentum_7d,arrival_price_ratio
1895193,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),5.0,1450.0,1650.0,1600.0,2006-03-21,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003123
1895194,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,1800.0,1900.0,0.0,2007-03-29,2200.0,...,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.600000
1895195,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),10.6,908.0,2318.0,2200.0,2007-03-29,1950.0,...,NaN,10.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004816
1895196,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,1900.0,2000.0,1950.0,2007-04-20,2033.0,...,NaN,10.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008560
1895197,Madhya Pradesh,Alirajpur,Alirajpur,Arhar (Tur-Red Gram)(Whole),16.7,2033.0,2033.0,2033.0,2007-04-20,1850.0,...,NaN,16.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008210


In [15]:
df = df.dropna()

print("Final Shape:", df.shape)

Final Shape: (2057003, 30)


In [16]:
os.makedirs(
    r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\data\processed",
    exist_ok=True
)

df.to_parquet(
    r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\data\processed\final_features_df.parquet",
    index=False
)

print("Feature parquet saved.")

Feature parquet saved.


In [17]:
FEATURES = [

    # categorical
    "state",
    "district",
    "market",
    "commodity",

    # market data
    "arrivals",
    "min_price",
    "max_price",

    # time
    "day",
    "month",
    "week",
    "quarter",
    "year",

    # cyclical
    "month_sin",
    "month_cos",

    # lag features
    "price_lag_1d",
    "price_lag_3d",
    "price_lag_7d",
    "price_lag_14d",

    # arrival lag features
    "arrival_lag_1d",
    "arrival_lag_7d",
    "arrival_lag_14d",

    # rolling stats
    "rolling_mean_7d",
    "rolling_mean_14d",

    "rolling_std_7d",
    "rolling_std_14d",

    # derived
    "price_momentum_7d",
    "arrival_price_ratio"
]

In [18]:
cat_cols = [
    "state",
    "district",
    "market",
    "commodity"
]

for col in cat_cols:

    df[col] = df[col].astype("category")

df.dtypes

state                        category
district                     category
market                       category
commodity                    category
arrivals                      float64
min_price                     float64
max_price                     float64
modal_price                   float64
date                   datetime64[ns]
target                        float64
day                             int32
month                           int32
week                            int64
quarter                         int32
year                            int32
month_sin                     float64
month_cos                     float64
price_lag_1d                  float64
price_lag_3d                  float64
price_lag_7d                  float64
price_lag_14d                 float64
arrival_lag_1d                float64
arrival_lag_7d                float64
arrival_lag_14d               float64
rolling_mean_7d               float64
rolling_std_7d                float64
rolling_mean

In [19]:
SPLIT_DATE = "2024-01-01"

train_df = df[
    df["date"] < SPLIT_DATE
]

test_df = df[
    df["date"] >= SPLIT_DATE
]

print("Train Shape:", train_df.shape)

print("Test Shape:", test_df.shape)

Train Shape: (2049394, 30)
Test Shape: (7609, 30)


In [20]:
X_train = train_df[FEATURES]

y_train = train_df["target"]

X_test = test_df[FEATURES]

y_test = test_df["target"]

print(X_train.shape)
print(X_test.shape)

(2049394, 27)
(7609, 27)


In [22]:
import lightgbm as lgb


In [23]:
model = lgb.LGBMRegressor(

    n_estimators=2000,

    learning_rate=0.03,

    max_depth=8,

    num_leaves=63,

    subsample=0.8,

    colsample_bytree=0.8,

    min_child_samples=20,

    reg_alpha=0.1,

    reg_lambda=0.1,

    random_state=42
)

In [24]:
model.fit(

    X_train,
    y_train,

    eval_set=[(X_test, y_test)],

    eval_metric="l1",

    categorical_feature=cat_cols,

    callbacks=[

        lgb.early_stopping(50),

        lgb.log_evaluation(100)

    ]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.311434 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4545
[LightGBM] [Info] Number of data points in the train set: 2049394, number of used features: 26
[LightGBM] [Info] Start training from score 2751.425237
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 176.361	valid_0's l2: 88761.8
Early stopping, best iteration is:
[146]	valid_0's l1: 165.189	valid_0's l2: 86288.9


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, max_depth=8,
              n_estimators=2000, num_leaves=63, random_state=42, reg_alpha=0.1,
              reg_lambda=0.1, subsample=0.8)

In [25]:
preds = model.predict(X_test)

preds[:10]

array([4281.17377521, 4461.639734  , 4500.26818767, 4432.97341911,
       4520.57085675, 4451.71925462, 4353.34034509, 4444.48630432,
       4383.49462517, 4422.24821929])

In [27]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score)


In [28]:
mae = mean_absolute_error(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

r2 = r2_score(
    y_test,
    preds
)

mask = y_test != 0

mape = np.mean(

    np.abs(
        (y_test[mask] - preds[mask])
        /
        y_test[mask]
    )

) * 100

In [29]:
print("\n========== MODEL PERFORMANCE ==========\n")

print("MAE  :", round(mae, 2))

print("RMSE :", round(rmse, 2))

print("R2   :", round(r2, 4))

print("MAPE :", round(mape, 2), "%")


========== MODEL PERFORMANCE ==========

MAE  : 165.19
RMSE : 293.75
R2   : 0.9467
MAPE : 4.52 %


In [30]:
importance_df = pd.DataFrame({

    "feature": FEATURES,

    "importance": model.feature_importances_

})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

importance_df.head(20)

,feature,importance
2,market,1170
6,max_price,1019
14,price_lag_1d,961
21,rolling_mean_7d,956
5,min_price,785
25,price_momentum_7d,715
15,price_lag_3d,510
3,commodity,463
23,rolling_std_7d,429
22,rolling_mean_14d,394


In [31]:
os.makedirs(
    r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\models",
    exist_ok=True
)

joblib.dump(

    model,

    r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\models\commodity_price_forecast_lgbm.pkl"
)

print("Model Saved Successfully.")

Model Saved Successfully.
